# Evaluation Jupyter Notebook

This notebook produces predictions, metrics, figures and misclassifications. 

It loads the artifacts created by `garbage_classifier_a2_talc.py` and produces:
- Loss curves + validation accuracy curves
- Classification report
- Confusion matrix
- Misclassification summary

### Expected folder
You should have an `outputs/` folder beside this notebook containing files like:
- `history.csv`
- `classification_report.txt`
- `confusion_matrix.npy` (optional) and/or `confusion_matrix.png`
- `test_predictions.csv`
- `misclassified.csv`


In [ ]:
# Config
# Point this to wherever you copied the TALC outputs folder.
OUT_DIR = "outputs"

# If you want this notebook to display misclassified images,
# set DATASET_ROOT to the local path that contains your dataset folders.
# Example (repo layout):
#   data/CVPR_2024_dataset_Test/Black/...
#
# If you don't have images locally, set DATASET_ROOT = None and you'll still
# get all plots + tables.
DATASET_ROOT = None  # e.g., "data"

# How many misclassified examples to display (if images are available)
SHOW_N_MIS = 12


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

def must_exist(path: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")
    return path

print("OUT_DIR:", OUT_DIR)
print("Files:", os.listdir(OUT_DIR))


## 1) Training Curves (from `history.csv`)

In [ ]:
hist_path = os.path.join(OUT_DIR, "history.csv")
must_exist(hist_path)
hist = pd.read_csv(hist_path)
hist

In [ ]:
# Loss curves
plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train_loss")
plt.plot(hist["epoch"], hist["val_loss"], label="val_loss")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss Curves")
plt.legend()
plt.tight_layout()
plt.show()

# Validation accuracy
plt.figure()
plt.plot(hist["epoch"], hist["val_acc"], label="val_acc")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Validation Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## 2) Classification Report

In [ ]:
report_txt_path = os.path.join(OUT_DIR, "classification_report.txt")
if os.path.exists(report_txt_path):
    print(open(report_txt_path, "r", encoding="utf-8").read())
else:
    print("No classification_report.txt found yet (that's okay). We'll recompute from predictions below.")


## 3) Confusion Matrix (load if saved, otherwise compute)

In [ ]:
# Load class names if available
class_names = None
class_names_path = os.path.join(OUT_DIR, "class_names.json")
if os.path.exists(class_names_path):
    class_names = json.load(open(class_names_path, "r", encoding="utf-8"))
    print("Loaded class_names:", class_names)

# Load confusion matrix if available
cm = None
cm_npy = os.path.join(OUT_DIR, "confusion_matrix.npy")
cm_png = os.path.join(OUT_DIR, "confusion_matrix.png")

if os.path.exists(cm_npy):
    cm = np.load(cm_npy)
    print("Loaded confusion_matrix.npy:", cm.shape)
elif os.path.exists(cm_png):
    img = plt.imread(cm_png)
    plt.figure()
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix (saved image)")
    plt.show()
else:
    print("No confusion_matrix.npy/png found yet. We'll compute from predictions next.")


## 4) Predictions + Metrics (from `test_predictions.csv`)

In [ ]:
pred_path = os.path.join(OUT_DIR, "test_predictions.csv")
must_exist(pred_path)
preds_df = pd.read_csv(pred_path)
preds_df.head()

In [ ]:
# Compute / print metrics from predictions (useful even if report file exists)
y_true = preds_df["true"].to_numpy()
y_pred = preds_df["pred"].to_numpy()

if class_names is None:
    if "true_name" in preds_df.columns:
        tmp = preds_df[["true", "true_name"]].drop_duplicates().sort_values("true")
        class_names = tmp["true_name"].tolist()
        print("Inferred class_names from CSV:", class_names)
    else:
        class_names = [str(i) for i in sorted(np.unique(y_true))]

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Compute confusion matrix if missing
if cm is None:
    cm = confusion_matrix(y_true, y_pred)

plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.colorbar()
plt.tight_layout()
plt.show()


## 5) Misclassifications

In [ ]:
mis_path = os.path.join(OUT_DIR, "misclassified.csv")
if os.path.exists(mis_path):
    mis_df = pd.read_csv(mis_path)
else:
    mis_df = preds_df[preds_df["true"] != preds_df["pred"]].copy()

print("Misclassified:", len(mis_df), "out of", len(preds_df))
mis_df.head()

In [ ]:
def resolve_path(p: str) -> str | None:
    """Try to map a TALC absolute path to a local dataset path."""
    if DATASET_ROOT is None:
        return None

    p = str(p)
    if os.path.exists(p):
        return p

    for key in ["CVPR_2024_dataset_Test", "CVPR_2024_dataset_Val", "CVPR_2024_dataset_Train"]:
        idx = p.find(key)
        if idx != -1:
            rel = p[idx:]  # e.g., CVPR_2024_dataset_Test/Blue/xyz.jpg
            cand = os.path.join(DATASET_ROOT, rel)
            if os.path.exists(cand):
                return cand

    # fallback: filename search (slow)
    fn = os.path.basename(p)
    for root, _, files in os.walk(DATASET_ROOT):
        if fn in files:
            return os.path.join(root, fn)

    return None


if DATASET_ROOT is None:
    print("DATASET_ROOT is None -> skipping image display. Set it to show misclassified images.")
else:
    show_df = mis_df.head(SHOW_N_MIS).copy()
    cols = 4
    rows = int(np.ceil(len(show_df) / cols))
    plt.figure(figsize=(4 * cols, 4 * rows))

    shown = 0
    for row in show_df.itertuples(index=False):
        local_path = resolve_path(getattr(row, "path", ""))
        if local_path is None:
            continue

        img = Image.open(local_path).convert("RGB")
        plt.subplot(rows, cols, shown + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"T:{row.true_name} / P:{row.pred_name}")
        shown += 1
        if shown >= rows * cols:
            break

    if shown == 0:
        print("Couldn't resolve any image paths. Check DATASET_ROOT.")
    else:
        plt.tight_layout()
        plt.show()
